# Finance Advisor Agent (No Memory)

Each query below is independent — no session/memory is attached, so the agent has no recollection of prior queries.

In [1]:
import os

from dotenv import load_dotenv
from openai import AsyncOpenAI

from agents import (
    Agent,
    ModelSettings,
    OpenAIChatCompletionsModel,
    Runner,
    set_tracing_disabled,
)

load_dotenv()

# The provided key is an OpenRouter key (sk-or-v1-...), so route the SDK's
# OpenAI-compatible client at OpenRouter instead of api.openai.com.
openrouter_client = AsyncOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)
set_tracing_disabled(True)  # tracing would otherwise POST to OpenAI's platform with this key

In [2]:
finance_advisor = Agent(
    name="Finance Advisor",
    instructions=(
        "You are a personal finance advisor. Track spending by category "
        "(e.g. groceries, transport, dining) from what the user tells you, "
        "and give clear, practical budget advice based on their stated budget. "
        "Be concise and specific with numbers."
    ),
    model=OpenAIChatCompletionsModel(
        model="openai/gpt-5-mini",
        openai_client=openrouter_client,
    ),
    # The OpenRouter account's credit balance can't cover the SDK's default
    # 65536 max_tokens, so cap output well under the account's affordable limit.
    # gpt-5-mini spends part of this budget on hidden reasoning tokens before
    # writing the visible answer, so too low a cap returns an empty final_output
    # (this happened at 1024 once the session carried extra history into the prompt).
    model_settings=ModelSettings(max_tokens=2048),
)

In [3]:
# Test query 1 — independent run, no memory of anything before this
result_1 = await Runner.run(
    finance_advisor,
    "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.",
)
print(result_1.final_output)

Thanks — I logged those. Summary for this week:
- Groceries: $120
- Uber (transport): $40
- Restaurants (dining out): $60
- Total: $220

Breakdown by percent: groceries 55%, Uber 18%, restaurants 27%. If this is a typical week, that’s about $880/month.

Quick, concrete options (numbers):
- Target weekly budget example (total $150/week): groceries $90, transport $30, dining $30. That would cut this week’s spending by $70 (saves $280/month if repeated).
- Quick wins:
  - Groceries: shave $30 → $120 to $90/wk = +$120/month. Meal plan, buy store brands, bulk staples.
  - Uber: cut $20 → $40 to $20/wk = +$80/month. Use public transit, combine trips, schedule rides during off-peak.
  - Restaurants: cut $30 → $60 to $30/wk = +$120/month. Limit eating out to one meal out/week or split portions.

Next step: what weekly or monthly budget do you want to stick to? I’ll track future entries and show progress vs. that target.


In [4]:
# Test query 2 — independent run; the agent has NOT retained query 1's context
result_2 = await Runner.run(
    finance_advisor,
    "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?",
)
print(result_2.final_output)

I don’t yet have your actual spending breakdown, so I can’t point to exact line items to trim. Here are concrete, actionable plans you can use next week to match a $250 weekly budget — pick one based on how much you want to cut.

Suggested baseline allocation (total $250):
- Groceries: $70
- Dining out / takeout: $35
- Transport (gas/public): $40
- Bills / subs (weekly share): $30
- Coffee / snacks: $15
- Entertainment / shopping: $30
- Misc / buffer: $30

If you need to cut next week, choose a level:

1) Small cut — save $25 (new total $225)
- Dining out: $35 → $25 (eat one fewer meal out; save $10)
- Coffee/snacks: $15 → $8 (bring coffee from home 3–4 days; save $7)
- Entertainment: $30 → $20 (pick 1 free/low-cost activity; save $10)

2) Medium cut — save $50 (new total $200)
- Dining out: $35 → $15 (cook lunches and 4 dinners at home; save $20)
- Groceries: $70 → $60 (plan meals, use a list, buy on-sale items; save $10)
- Coffee/snacks: $15 → $5 (bring coffee every day; save $10)
- 

# Task 3: Finance Advisor Agent (With Memory — SQLiteSession)

Same agent, same model (`openai/gpt-5-mini` via OpenRouter) as the no-memory section above — the only difference is a `SQLiteSession` attached to `Runner.run`. The session stores conversation history automatically and feeds it back on the next call, so the second query can actually use the expense breakdown from the first.

In [5]:
from agents import SQLiteSession

# Reuses the same `finance_advisor` agent defined above (same model, same instructions).
# In-memory-only DB — pass db_path="./conversations.db" instead to persist across kernel restarts.
memory_session = SQLiteSession("finance_advisor_demo")

In [6]:
# Test query 1 — same wording as the no-memory run, now attached to a session
result_1_mem = await Runner.run(
    finance_advisor,
    "I spent $120 on groceries, $40 on Uber, and $60 on restaurants this week.",
    session=memory_session,
)
print(result_1_mem.final_output)

Got it — I logged this week as:
- Groceries: $120
- Uber/transport: $40
- Restaurants/dining out: $60
Total this week: $220.

If that repeats each week (×4), that’s about $880/month (Groceries $480, Uber $160, Restaurants $240).

Practical targets and savings (concrete numbers)
- New weekly caps: Groceries $100, Uber $30, Restaurants $40.
  - Monthly result (×4): Groceries $400, Uber $120, Restaurants $160 = $680/month.
  - Estimated monthly savings vs current pattern: $880 − $680 = $200/month.

How to hit those caps (quick tactics)
- Groceries: meal plan, buy staples on sale, use a list, cook bigger batches ($20/week saved).
- Uber: combine errands, use transit/carpool for short trips, book pooled rides when possible ($10/week).
- Restaurants: limit to 1–2 meals out/week, pick lower-cost options or split entrees ($20/week).

Next step
Tell me your weekly or monthly budget and household size (are these typical weeks?). I’ll adjust targets to match your goals and keep tracking.


In [7]:
# Test query 2 — same wording as the no-memory run; this time the session carries
# query 1's expense breakdown forward, so the agent can reference it directly.
result_2_mem = await Runner.run(
    finance_advisor,
    "My weekly budget is $250. Based on everything I told you so far, where can I cut spending next week?",
    session=memory_session,
)
print(result_2_mem.final_output)

Summary — current week
- Logged: Groceries $120, Uber $40, Restaurants $60 = $220 this week.
- Weekly budget: $250 → you’re already $30 under.

If you want to cut more next week, pick one of these realistic plans (with exact targets and quick tactics):

1) Small / maintain buffer
- Targets: Groceries $120, Uber $35, Restaurants $55 = $210
- Savings vs current: $10; vs budget: $40 spare
- Tactics: use one promo/pooled Uber ride, skip one appetizer when dining out.

2) Moderate / steady savings (recommended)
- Targets: Groceries $100, Uber $30, Restaurants $40 = $170
- Savings vs current: $50/week → $200/month
- Tactics: meal-plan 5 dinners + 1 batch-cook, take transit or walk 1–2 short trips, limit eating out to 1 takeout/meal.

3) Aggressive / fast savings
- Targets: Groceries $90, Uber $20, Restaurants $30 = $140
- Savings vs current: $80/week → $320/month
- Tactics: cook almost all meals (use leftovers), only book pooled/discounted rides or transit, allow 1 low-cost meal out.

Which 

## Side-by-Side: Without Memory vs. With Memory

Both runs answer the same second query ("Based on everything I told you so far..."). Without memory, the agent has no record of query 1 and falls back to generic advice. With `SQLiteSession`, it recalls the exact $120/$40/$60 breakdown and can recommend specific, targeted cuts.

In [8]:
print("=== WITHOUT MEMORY (independent runs) ===")
print(result_2.final_output)

print("\n" + "=" * 60 + "\n")

print("=== WITH MEMORY (SQLiteSession) ===")
print(result_2_mem.final_output)

=== WITHOUT MEMORY (independent runs) ===
I don’t yet have your actual spending breakdown, so I can’t point to exact line items to trim. Here are concrete, actionable plans you can use next week to match a $250 weekly budget — pick one based on how much you want to cut.

Suggested baseline allocation (total $250):
- Groceries: $70
- Dining out / takeout: $35
- Transport (gas/public): $40
- Bills / subs (weekly share): $30
- Coffee / snacks: $15
- Entertainment / shopping: $30
- Misc / buffer: $30

If you need to cut next week, choose a level:

1) Small cut — save $25 (new total $225)
- Dining out: $35 → $25 (eat one fewer meal out; save $10)
- Coffee/snacks: $15 → $8 (bring coffee from home 3–4 days; save $7)
- Entertainment: $30 → $20 (pick 1 free/low-cost activity; save $10)

2) Medium cut — save $50 (new total $200)
- Dining out: $35 → $15 (cook lunches and 4 dinners at home; save $20)
- Groceries: $70 → $60 (plan meals, use a list, buy on-sale items; save $10)
- Coffee/snacks: $15 